In [1]:
import vtk
from pathlib import Path
from scipy.stats import multivariate_normal
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy import io as scio
from scipy import stats as sci_stats
import pickle
import scipy.spatial as sp
from typing import Union,Dict,List
import os
import SimpleITK as sitk
import numpy as np
import sys
sys.path.append('../scripts')
from gpa import gpa

In [2]:
#variables
path_to_processed_cohort = Path(r"/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/mouse_skull_workdir")


In [ ]:
def compute_center_of_mesh(mesh):
    centerOfMassFilter = vtk.vtkCenterOfMass()
    centerOfMassFilter.SetInputData(mesh)
    centerOfMassFilter.SetUseScalarsAsWeights(False)
    centerOfMassFilter.Update();
    center = centerOfMassFilter.GetCenter()
    return center




reader = vtk.vtkOBJReader()
num_of_points = 0
meshes_dict = {}
for path in tqdm(list(Path(path_to_processed_cohort).iterdir())):
    mesh_file = path / "head_affine_registered.obj"
    if not mesh_file.exists():
        continue
    reader.SetFileName(mesh_file)
    reader.Update()
    mesh = reader.GetOutput()
    points = mesh.GetPoints()
    center_of_mesh = np.array( compute_center_of_mesh(mesh) )
    num_of_points = points.GetNumberOfPoints()
    res_mesh = []
    for i in range(points.GetNumberOfPoints()):
        pt = points.GetPoint(i)
        res_mesh.append(list( np.array( pt) - center_of_mesh ))
    meshes_dict[str( mesh_file.parent.name ) ] = res_mesh


  0%|          | 0/769 [00:00<?, ?it/s]

In [4]:
def compute_outliers(meshes_dict: Union[Dict[str,np.ndarray],List[np.ndarray]]):
    
    stats = []
    index_sum=None
    if isinstance(meshes_dict,list):
        num_of_points = len( meshes_dict[0] )
        working_dict = list(zip(range(num_of_points),meshes_dict))
        
    else:
        num_of_points = len( list(meshes_dict.values())[0] )
        working_dict = list(meshes_dict.items())
            
    for i in tqdm(range(num_of_points)): # i in range of num_of_points
        index = []
        points = []
        #print (i, working_dict)
        for key, mesh_v in working_dict:
            index.append(key)
            points.append(mesh_v[i])
        
        df = pd.DataFrame( data = points, index=index, columns=['x','y','z'])
        
        indexes = (np.abs(sci_stats.zscore(df)) < 2).all(axis=1)
        
        if index_sum is None:
            index_sum = (~indexes).astype(int)
        else:
            index_sum += (~indexes).astype(int)
        df = df[indexes]
        stats.append(df.describe())
    
    return stats, index_sum

    
_,_,res_meshes = gpa(meshes_dict)
stats, index_sum = compute_outliers(res_meshes)

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/732 [00:00<?, ?it/s]

procrustes distance  0.00043563311389365893


  0%|          | 0/732 [00:00<?, ?it/s]

procrustes distance  8.124042342432704e-10


  0%|          | 0/733 [00:00<?, ?it/s]

  0%|          | 0/25680 [00:00<?, ?it/s]

In [ ]:
def flip_image(im_file_in: str, im_file_out: str):
    # Load your image (replace 'image_path' with the path to your image)
    image = sitk.ReadImage(im_file_in)

    # Flip the image along a specific axis (e.g., along the x-axis)
    # Flip the image along the x-axis
    #flipped_image = sitk.Flip(image, [True, False, False])  # [X, Y, Z] axis flipping
    # Specify the axis along which you want to mirror (0 for x-axis, 1 for y-axis, 2 for z-axis)
    axis_to_mirror = 0  # For example, mirror along the x-axis
    image_array = sitk.GetArrayFromImage(image)
    # Flip the image along the specified axis
    mirrored_image_array = np.flip(image_array, axis=(1,2))

    # Create a SimpleITK image from the mirrored NumPy array
    flipped_image = sitk.GetImageFromArray(mirrored_image_array)

    # Set the origin, spacing, and direction cosine matrix if needed
    flipped_image.SetOrigin(image.GetOrigin())
    flipped_image.SetSpacing(image.GetSpacing())
    flipped_image.SetDirection(image.GetDirection())

    # Apply the flipping transformation to the image

    sitk.WriteImage(flipped_image, im_file_out) 


##remove files

for k,v in tqdm(sorted(index_sum.items(), key=lambda x:x[1])):
    
    if v > 10000:
        folder = path_to_processed_cohort / k

        try:
            flip_image(folder / "ct_image_.nii.gz", folder / "oriented_image.nii.gz")
            flip_image(folder / "bone_coregistration_mask.nii.gz", folder / "oriented_mask.nii.gz")
            os.remove( folder / "affine_head.mat" )
            os.remove( folder / "coregistered_head.nii.gz" )
            #os.remove( folder / "oriented_image.nii.gz" )
            #os.remove( folder / "oriented_mask.nii.gz" )
            #os.remove( folder / "bone_coregistration_mask.nii.gz" )
            os.remove( folder / "head_affine_registered.obj" )
            os.remove( folder / "nl_transformation_params.txt" )
        except:
            pass
        






In [ ]:
flip_transform = sitk.ScaleTransform(3)
flip_transform.SetScale((-1, 1, 1))  # Flip along the x-axis

# Apply the transformation to the image
flipped_image = sitk.Resample(image, flip_transform)


In [ ]:
image.GetOrigin()

In [ ]:
flipped_image.GetDirection()

In [ ]:
image.TransformContinuousIndexToPhysicalPoint([10,10,10])


In [ ]:
flipped_image.TransformContinuousIndexToPhysicalPoint([10,10,10])

In [ ]:
axis_to_mirror = 0  # For example, mirror along the x-axis

# Use the Mirror function to mirror the image
mirrored_image = sitk.Mirror(image, [1, 0, 0][axis_to_mirror])


In [ ]:
_,_,res_meshes = gpa(meshes_dict)

In [ ]:
res_meshes

In [ ]:
stats, index_sum = compute_outliers(res_meshes)

In [5]:
list(sorted(index_sum.items(), key=lambda x:x[1]))

[('wt_M_82Z-CF279', 0),
 ('wt_F_82Z-09277', 0),
 ('wt_F_82Z-09858', 0),
 ('wt_F_82Z-11895', 0),
 ('wt_M_82Z-CF432', 0),
 ('wt_M_82Z-11210', 0),
 ('wt_M_82Z-07519', 0),
 ('wt_M_82Z-09037', 0),
 ('wt_F_82Z-07741', 1),
 ('wt_M_82Z-CG965', 2),
 ('wt_F_82Z-CE573', 3),
 ('wt_F_82Z-11888', 4),
 ('wt_F_82Z-CG969', 4),
 ('wt_F_82Z-CF284', 4),
 ('wt_F_82Z-07938', 4),
 ('wt_F_82Z-08985', 4),
 ('wt_M_82Z-08477', 5),
 ('wt_F_82Z-07489', 5),
 ('wt_M_82Z-10968', 5),
 ('wt_M_82Z-11782', 6),
 ('wt_F_82Z-11535', 6),
 ('wt_M_82Z-11169', 6),
 ('wt_M_82Z-CE569', 6),
 ('wt_M_82Z-CG673', 8),
 ('wt_M_82Z-10635', 9),
 ('wt_F_82Z-12143', 10),
 ('wt_M_82Z-11889', 10),
 ('wt_M_82Z-12084', 11),
 ('wt_M_82Z-07518', 12),
 ('wt_M_82Z-09391', 12),
 ('wt_F_82Z-07085', 13),
 ('wt_M_82Z-CF431', 13),
 ('wt_M_82Z-09034', 13),
 ('wt_F_82Z-07528', 14),
 ('wt_F_82Z-11933', 14),
 ('wt_F_82Z-07610', 15),
 ('wt_M_82Z-09869', 15),
 ('wt_F_82Z-07311', 16),
 ('wt_M_82Z-08807', 16),
 ('wt_M_82Z-09591', 17),
 ('wt_M_82Z-09870', 17),


In [ ]:
len(list(sorted(index_sum.items(), key=lambda x:x[1])))
